# C.P. Cavafy — Workshop Notebook
**CompLit 126x — Love in Context**

This notebook implements the prompt chain designed by the Cavafy workshop group. Their chain used a **quality sliders + persona generation** architecture — turning the model's binary checklist of traits into adjustable sliders, then building an author persona to generate from.

```
one-shot       ──▶  identify key      ──▶  reframe as
sonnet              qualities +             sliders (0–100)
                    justify match             │
                                              ▼
                                        inspect model's
                                        internal steps
                                              │
                                              ▼
                                        choose "Type B":
                                        personal pronouns,
                                        sense of place
                                              │
                                              ▼
                                        minus 20%
                                        aggression
                                              │
                                              ▼
generate       ──▶  write AS persona  ◀─  refined poem
author persona      (don't explicitly
                     signal traits
                     IN the poem)
```

**Qualities the group identified:** direct address / "you", melancholy, twist, escalates quickly, doesn't adhere strictly to sonnet conventions.

**What makes this chain interesting:** The group noticed that the model treated Cavafy's qualities as a binary checklist — either present or absent. When asked to revise, it would swing to extremes: adding a quality fully or removing it entirely. Reframing qualities as *sliders with percentages* gave the group fine-grained control (e.g., "minus 20% aggression" instead of "less aggressive"). The persona step at the end was experimental: instead of prompting *about* Cavafy, they asked the model to *become* Cavafy and write naturally — holding the persona but not explicitly signaling traits in the poem.

---
**Run the cells in order.** Each step builds on the previous one.

## Setup
Run the two cells below once at the start of your session.

In [ ]:
# Install the OpenAI SDK (run once per session)
%pip install openai --quiet
print("✓ Installed")

In [ ]:
from openai import OpenAI
import json
import os

# ── API Key ──────────────────────────────────────────────────────────────────
# In Google Colab:
#   1. Click the 🔑 (Secrets) icon in the left sidebar
#   2. Add a secret named  OPENAI_API_KEY  with your key
#   3. Toggle "Notebook access" to ON, then run this cell
#
# Locally: set the OPENAI_API_KEY environment variable

try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
    print("✓ Using Colab Secrets")
except (ImportError, Exception):
    api_key = os.environ.get('OPENAI_API_KEY')
    print("✓ Using environment variable")

client = OpenAI(api_key=api_key)
MODEL = "gpt-4o"

# Helper: call the model and return the text
def ask(prompt, system=None):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    response = client.responses.create(model=MODEL, input=messages)
    return response.output_text

print(f"✓ Client ready | Model: {MODEL}")

---
## Step 1: One-Shot (Baseline)

Start with a minimal prompt — just ask for a Cavafy-style sonnet. This establishes our baseline before we start adjusting.

In [ ]:
# ── Step 1: One-shot ─────────────────────────────────────────────────────────

one_shot = ask(
    "Generate a sonnet in the style of C.P. Cavafy."
)

print("ONE-SHOT RESULT")
print("═" * 60)
print(one_shot)
print("\n" + "═" * 60)
print("\n↓ The model will try to hit Cavafy's notes, but probably")
print("  treats his qualities as a checklist: all or nothing.")

---
## Step 2: Identify Key Qualities + Justify

Now we ask the model to name Cavafy's key qualities, then evaluate how well the one-shot actually matches them. The group found that the model could *identify* the right qualities but applied them like a checklist — checking every box at once rather than balancing them.

In [ ]:
# ── Step 2: Identify qualities and justify ───────────────────────────────────

qualities = ask(
    f"""Here is a sonnet generated in the style of C.P. Cavafy:

{one_shot}

Identify the key qualities of Cavafy's poetry. The group working on
this chain identified these as starting points — but you may add others:

- **Direct address / "you"**: Cavafy often speaks to a "you" — a lover,
  a younger self, the reader
- **Melancholy**: a pervasive tone of loss, regret, and longing
- **Twist**: a sudden turn or revelation, often in the final lines
- **Escalates quickly**: the emotional or narrative stakes rise fast
- **Loose sonnet form**: doesn't adhere strictly to sonnet conventions

For each quality, evaluate how well the poem above captures it.
Quote specific lines as evidence.

Format each quality as:
**[Quality Name]**: [description]
- Match: [how well the poem captures this, with quoted evidence]"""
)

print("KEY QUALITIES + JUSTIFICATION")
print("═" * 60)
print(qualities)

---
## Step 3: Reframe as Sliders

This is the group's central insight. Instead of treating qualities as binary (present/absent), we put each one on a **0–100 scale** and score the current poem. This gives us something we can *adjust by degree* rather than toggling on and off.

The problem with checklists: when you say "add more nostalgia," the model maxes it out. When you say "less formality," it drops to zero. Sliders let you say "increase nostalgia from 40 to 65" — a much more useful instruction.

In [ ]:
# ── Step 3: Reframe as sliders ───────────────────────────────────────────────

sliders = ask(
    f"""Here are the key qualities of Cavafy's poetry that you identified:

{qualities}

Now reframe each quality as a slider on a 0–100 scale, where:
- 0 = completely absent
- 50 = moderate / balanced
- 100 = dominant / overwhelming

For each slider:
1. Name the quality
2. Describe what 0, 50, and 100 would look like in a poem
3. Score the one-shot poem on this scale
4. Score where you think Cavafy's actual poems typically sit

Format as a table or list. The gap between the one-shot score and
Cavafy's actual score tells us what to adjust."""
)

print("QUALITY SLIDERS")
print("═" * 60)
print(sliders)

---
## Step 4: Inspect the Model's Process

The group's next move: ask the model to reveal its *own* step-by-step process for generating the poem. They discovered that the model made decisions automatically — choosing a particular framing, subject, or "type" of Cavafy poem — that constrained everything after it. By making those defaults visible, they could redirect.

This is a form of **process transparency**: instead of just critiquing the output, you critique the *generation process itself*.

In [ ]:
# ── Step 4: Inspect the model's process ──────────────────────────────────────

process = ask(
    f"""You just wrote this poem in the style of C.P. Cavafy:

{one_shot}

And here are the quality sliders you scored it on:

{sliders}

Walk me through your step-by-step process for generating this poem.
Be specific about the decisions you made at each stage:

1. What was your FIRST decision? (subject, setting, situation)
2. What "type" of Cavafy poem did you default to? (e.g., historical
   reflection, erotic memory, philosophical meditation, city scene)
3. How did you decide on the tone and emotional register?
4. What Cavafy-specific moves did you consciously apply?
5. What did you prioritize, and what did you sacrifice?

Be honest about defaults and shortcuts. Where did you fall back on
generic "Cavafy-ish" moves instead of something more specific?"""
)

print("MODEL'S GENERATION PROCESS")
print("═" * 60)
print(process)
print("\n" + "═" * 60)
print("\n↓ Look at Step 1 — what 'type' of Cavafy poem did it default to?")
print("  The group chose to redirect toward 'Type B': personal pronouns,")
print("  direct address, and a strong sense of place.")

---
## Step 5: Choose "Type B" — Personal Pronouns + Sense of Place

After seeing the model's default process, the group redirected it. Instead of the model's default "type" of Cavafy poem (probably historical/elegiac), they asked for what they called **"Type B"**: a poem built around personal pronouns and direct address ("you"), with a strong sense of place.

This is the Cavafy who writes *to* someone — not about the fall of empires, but about the specific room, the specific body, the specific city street.

In [ ]:
# ── Step 5: Type B — personal pronouns + sense of place ──────────────────────

type_b_poem = ask(
    f"""Generate a new poem in the style of C.P. Cavafy.

Here are the quality sliders to target:

{sliders}

But this time, take a different approach than your default. Specifically:

TYPE B CONSTRAINTS:
- Use personal pronouns throughout — address a "you" directly,
  as Cavafy does in poems like "Body, Remember..." or "One Night"
- Build a strong sense of place — a specific room, street, city,
  café. Not abstract space but somewhere you can smell and hear.
- Keep the melancholy and the twist, but ground them in the
  intimate and the local, not the historical or mythic.

The poem should feel like Cavafy speaking quietly to someone
he once knew, in a place they both remember.

Write only the poem."""
)

print("TYPE B POEM")
print("═" * 60)
print(type_b_poem)

---
## Step 6: Minus 20% Aggression

The group's fine-tuning move. After generating the Type B poem, they found it was still too forceful — the model was *pushing* Cavafy's qualities too hard, making them feel deliberate rather than natural. Their instruction: **minus 20% aggression.**

This is the slider concept in action. Not "less intense" (which could mean anything), but a specific percentage reduction in a named quality. The model can calibrate to that.

In [ ]:
# ── Step 6: Minus 20% aggression ─────────────────────────────────────────────

refined_poem = ask(
    f"""Here is a poem in the style of C.P. Cavafy:

{type_b_poem}

This poem is pushing too hard. The Cavafy qualities are there but
they feel forced — like the poem is trying to prove it's a Cavafy poem
instead of just being one.

Revise with this adjustment: **minus 20% aggression.**

Specifically:
- Soften any line that announces its own theme
- Where the poem declares, make it suggest instead
- Where it escalates too fast, let it linger a beat longer
- Reduce the intensity by about 20% across the board — the melancholy
  should ache, not shout; the desire should be remembered, not performed

Keep everything else: the personal pronouns, the sense of place,
the direct address, the twist. Just turn the volume down slightly.

Write only the revised poem."""
)

print("REFINED POEM (−20% aggression)")
print("═" * 60)
print(refined_poem)

---
## Step 7: Generate Author Persona

The group's final experimental move: instead of prompting *about* Cavafy, ask the model to construct a detailed **persona** of the author — his habits of mind, his way of seeing, his relationship to language — and then hold that persona while writing.

The key instruction: write as this persona *without explicitly signaling the traits*. Don't announce "I am nostalgic" — just be nostalgic. This is the difference between a poet and a checklist.

In [ ]:
# ── Step 7a: Build the persona ───────────────────────────────────────────────

persona = ask(
    f"""Based on everything we've discussed about C.P. Cavafy's poetry,
construct a detailed author persona. This should read like a character
study — not a Wikipedia biography, but a portrait of his *mind as a
poet*. Include:

- How he sees the world (what he notices, what he ignores)
- His relationship to memory, desire, and time
- His emotional habits (what he permits himself to feel on the page)
- His formal instincts (how he builds a poem, where he lingers,
  where he cuts)
- His characteristic moves (the turn he always makes, the thing
  he always withholds)
- The gap between what he says and what he means

Write this as a second-person address: "You are..." — as if briefing
an actor about to play the role.

Here are the quality sliders for reference:

{sliders}"""
)

print("CAVAFY PERSONA")
print("═" * 60)
print(persona)

In [ ]:
# ── Step 7b: Write from persona ──────────────────────────────────────────────
# The persona becomes the system prompt. The model writes AS Cavafy,
# not ABOUT Cavafy.

persona_poem = ask(
    """Write a new poem. Do not announce your themes or signal your
traits explicitly — no line should read like a thesis statement about
what kind of poet you are. Just write naturally, as yourself.

The poem should feel like a new discovery, not an imitation. Ground
it in a specific moment — something seen, someone remembered — and
let everything else emerge from there.

Write only the poem. No title, no explanation.""",
    system=persona  # The persona IS the system prompt
)

print("PERSONA POEM")
print("═" * 60)
print(persona_poem)
print("\n" + "═" * 60)
print("\n↓ Compare this to the one-shot. The persona poem should feel")
print("  less like an imitation and more like a voice — the difference")
print("  between 'a poem about Cavafy's themes' and 'a Cavafy poem.'")

---
## Compare All Versions

Now look at the full progression. Each step changed something specific — and the *kind* of change matters as much as whether it's "better."

In [ ]:
# ── Side-by-side comparison ──────────────────────────────────────────────────

print("PROGRESSION")
print("\n" + "═" * 60)
print("1. ONE-SHOT (no context)")
print("═" * 60)
print(one_shot)

print("\n" + "═" * 60)
print("2. TYPE B (personal pronouns + sense of place)")
print("═" * 60)
print(type_b_poem)

print("\n" + "═" * 60)
print("3. REFINED (−20% aggression)")
print("═" * 60)
print(refined_poem)

print("\n" + "═" * 60)
print("4. PERSONA (wrote AS Cavafy, not ABOUT Cavafy)")
print("═" * 60)
print(persona_poem)

print("\n" + "═" * 60)
print("\nFor your essay, consider:")
print("  → What did the slider framing change vs. the binary checklist?")
print("  → What happened when you chose 'Type B' over the model's default?")
print("  → What did '−20% aggression' actually change in the language?")
print("  → Does the persona poem feel different from the refined one?")
print("  → What's the difference between imitating traits and inhabiting a voice?")

---
## Going Further

This chain is a starting point. Here are ways to extend it for your assignment:

**Feed in actual poems.** The group's chain didn't include Cavafy's actual text. Try adding 3–5 poems ("Ithaka," "The City," "One Night," "Body, Remember...") and re-running the slider analysis. Do the scores change?

**Fine-tune the sliders.** Run Steps 3–4 again with different percentage adjustments. What happens if you push one quality to an extreme while holding the others steady?

**Try different Step 1 overrides.** The corrected poem used a mundane present-tense encounter. What if you started with a historical scene? A dream? A letter never sent?

**Layer multiple personas.** Generate personas for two different poets and blend them — 70% Cavafy, 30% someone else. What emerges?

**Generate love song lyrics.** Swap the generation prompt to ask for song lyrics — verse, chorus, bridge — instead of a poem. How does the model handle Cavafy's understatement in a form that usually demands directness?

**Submitting your work:**
- **Lyrics**: Submit an album's worth of songs, with your favorite first
- **Audio**: Take your best lyrics to [Suno](https://suno.com) and generate audio
- **Essay** (500–700 words): Explain your prompt chain, include sample prompts, and reflect on what GPT-4o got right and wrong about your poet